# 03 — Benchmarks: Matmul & Conv (RTX 3050)

Run PyTorch vs Triton matmul and torch vs extension vs Triton conv. **Run from repo root** so imports resolve.

In [ ]:
import sys
import time
from pathlib import Path
root = Path("..").resolve()
sys.path.insert(0, str(root))
import torch

## Matmul: PyTorch vs Triton

In [ ]:
def bench_matmul(N, dtype=torch.float16, repeat=50):
    A = torch.randn(N, N, device="cuda", dtype=dtype)
    B = torch.randn(N, N, device="cuda", dtype=dtype)
    for _ in range(10):
        torch.matmul(A, B)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeat):
        torch.matmul(A, B)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000 / repeat

if torch.cuda.is_available():
    gflops = lambda N, ms: 2*N**3/1e9/(ms/1000)
    for N in [512, 1024, 2048]:
        ms = bench_matmul(N)
        print(f"Matmul {N}x{N} PyTorch: {ms:.4f} ms, {gflops(N,ms):.1f} GFLOPS")
    try:
        from triton_kernels.matmul import matmul_triton
        for N in [512, 1024, 2048]:
            A = torch.randn(N, N, device="cuda", dtype=torch.float16)
            B = torch.randn(N, N, device="cuda", dtype=torch.float16)
            for _ in range(10): matmul_triton(A, B)
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(50): matmul_triton(A, B)
            torch.cuda.synchronize()
            ms = (time.perf_counter() - t0) * 1000 / 50
            print(f"Matmul {N}x{N} Triton:  {ms:.4f} ms, {gflops(N,ms):.1f} GFLOPS")
    except ImportError as e:
        print("Triton skipped:", e)

## Conv: torch vs custom vs Triton

In [ ]:
if torch.cuda.is_available():
    import torch.nn as nn
    B, C_in, H, W = 128, 1, 28, 28
    C_out = 32
    conv = nn.Conv2d(C_in, C_out, 3, padding=0).cuda()
    x = torch.randn(B, C_in, H, W, device="cuda")
    for _ in range(10): _ = conv(x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(50): _ = conv(x)
    torch.cuda.synchronize()
    t_torch = (time.perf_counter() - t0) * 1000 / 50
    print(f"torch.nn.Conv2d: {t_torch:.4f} ms")
    try:
        import custom_conv
        w, b = conv.weight, conv.bias
        t0 = time.perf_counter()
        for _ in range(50): _ = custom_conv.custom_conv2d(x, w, b)[0]
        torch.cuda.synchronize()
        t_custom = (time.perf_counter() - t0) * 1000 / 50
        print(f"custom_conv2d:   {t_custom:.4f} ms ({t_torch/t_custom:.2f}x)")
    except ImportError:
        print("custom_conv not installed")
    try:
        from triton_kernels.conv import conv2d_triton_fp16
        x_h, w_h = x.half(), conv.weight.half()
        b_h = conv.bias.half()
        t0 = time.perf_counter()
        for _ in range(50): _ = conv2d_triton_fp16(x_h, w_h, b_h)
        torch.cuda.synchronize()
        t_tr = (time.perf_counter() - t0) * 1000 / 50
        print(f"Triton conv:     {t_tr:.4f} ms ({t_torch/t_tr:.2f}x)")
    except ImportError as e:
        print("Triton conv skipped:", e)